# 03 · CKINN: constraint strength and behaviour (RICH only)
CKINN = the MLP of notebook 02 plus the soft clinical-constraint penalty. Here we choose lambda on **validation data only**, then compare MLP, CKINN and the monotone GBM on a held-out RICH split.

In [ ]:
import os, sys, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import ckinn_lib as L

DATA = pathlib.Path(os.environ.get("CKINN_DATA", "data"))        # harmonized CSVs from notebook 01
OUT = pathlib.Path("results"); OUT.mkdir(exist_ok=True)
FIG = pathlib.Path("figs"); FIG.mkdir(exist_ok=True)
FAST = os.environ.get("CKINN_FAST", "0") == "1"                    # FAST=1: small subsample, code check only
SEED = 0
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
rich, nag = L.load_data(DATA)
if FAST:
    rich = rich.sample(40000, random_state=SEED)                   # development cohort only
print(f"RICH {len(rich):,} rows, {rich.event.sum():,} events | FAST={FAST}")

## 1. Configuration

In [ ]:
TIER = "T3"
cols = L.TIERS[TIER]
LAMBDAS = [0, 0.1, 1.0] if FAST else [0, 0.01, 0.1, 1.0, 10.0]
EPOCHS = 6 if FAST else 40
TOL = 0.002          # pre-specified rule: choose the largest lambda whose validation scaled Brier is within TOL of the best
train, test = next(L.cv_folds(rich, 5, SEED))
fit_part, val = next(L.cv_folds(train, 5, SEED + 1))
print(len(fit_part), len(val), len(test))

## 2. Lambda sweep on validation data

In [ ]:
rows = []
for lam in LAMBDAS:
    f = L.fit_with_baseline(L.DeepCox(cols, lam=lam, epochs=EPOCHS, seed=SEED), fit_part)
    r = L.evaluate(f, val); r.update(lam=lam); rows.append(r)
    print(f"lambda={lam}: scaled Brier {r['scaled_brier']:.4f}, violation {r['viol_rate']:.3f}")
sweep = pd.DataFrame(rows)[["lam", "cindex", "scaled_brier", "oe", "cal_slope", "viol_rate"]]
sweep.to_csv(OUT / "03_lambda_sweep_validation.csv", index=False)
best = sweep.scaled_brier.max()
LAM = float(sweep[sweep.scaled_brier >= best - TOL].lam.max())
print("Chosen lambda:", LAM)
sweep.round(4)

## 3. Refit on the full training split and test

In [ ]:
specs = {"MLP (lambda=0)": lambda: L.DeepCox(cols, lam=0.0, epochs=EPOCHS, seed=SEED),
         f"CKINN (lambda={LAM})": lambda: L.DeepCox(cols, lam=LAM, epochs=EPOCHS, seed=SEED),
         "XGB-Cox monotone": lambda: L.XGBCox(cols, True, 100 if FAST else 300, seed=SEED)}
res, fits = L.run_models(specs, train, test)
res.to_csv(OUT / "03_ckinn_vs_baselines_rich_holdout.csv", index=False)
res[["model", "cindex", "auc_3y", "scaled_brier", "oe", "cal_slope", "viol_rate"]].round(3)

## 4. Partial-dependence curves: does the learned risk surface respect the priors?

In [ ]:
def pdc(model, df, var, n=1500):
    s = df.sample(n, random_state=SEED).reset_index(drop=True)
    grid = np.linspace(df[var].quantile(0.02), df[var].quantile(0.98), 25)
    out = []
    for g in grid:
        p = s.copy(); p[var] = g
        out.append(model.lp(p).mean())
    return grid, np.array(out) - out[0]
show = ["age", "bmi", "tg", "fpg"]
fig, axes = plt.subplots(1, len(show), figsize=(15, 3.6))
for ax, v in zip(axes, show):
    for name, f in fits.items():
        g, y = pdc(f.model, test, v); ax.plot(g, y, label=name)
    ax.set_xlabel(v); ax.set_title(f"prior: {'increasing' if L.PRIORS[v] > 0 else 'decreasing'}")
axes[0].set_ylabel("relative log-hazard"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "03_partial_dependence.png"); plt.show()

## 5. Calibration at 3 years

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.4))
for name, f in fits.items():
    ct = L.calibration_table(f, test).dropna()
    ax.plot(ct.pred * 100, ct.obs * 100, "o-", label=name)
m = max(ax.get_xlim()[1], ax.get_ylim()[1]); ax.plot([0, m], [0, m], "k--", lw=0.8)
ax.set_xlabel("predicted 3-year risk, %"); ax.set_ylabel("observed (Kaplan-Meier), %"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "03_calibration.png"); plt.show()

**Decision for the protocol.** Record the chosen lambda and the rule used in the decision log before running notebook 04. Do not re-tune lambda after seeing transport results.